## Plot correlation between following errors and elevation angle

##### **Description**


Struts 5 and 6 in the M2 Hexapod support most of the M2 weight when the telescope is at the horizon position [(see Figure 2 on this page)](https://ts-mthexapod.lsst.io/algorithm/kinematics.html). This load might add a correlation between the faults and the elevation angle. 

We want a histogram showing the number of Following Error faults per elevation angle (if possible on Strut 5 and on Strut 6). We can bin the elevation angles for every 5º. 

LSSTCam data can be analysed. ComCam data too, however with some missing data for some dates.
The challenge here will be to build a query to the EFD that can query the faults without downloading too much data.). This load might add a correlation between the faults and the elevation angle. 


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
from datetime import timedelta
from matplotlib import pyplot as plt

from astropy.time import Time

In [ ]:
import numpy as np

In [ ]:
from lsst.summit.utils.efdUtils import (
    getEfdData,
    makeEfdClient,
)


## MTHexapod Following Error

This notebook will print out the log messages with `min_log_level` for a given `day_obs`.\
The `sal_index` can be either 1 (Camera Hexapod) or 2 (M2 Hexapod).  
The `min_log_level` correspondts to the minimum logging level required to print the messages.\
See the [Logging Levels](https://docs.python.org/3/library/logging.html#levels) page for more details.


In [ ]:
# Initialize an EFD client
efd_client = makeEfdClient()


def query_MTMount_elevation_telemetry(client, t_stamp, delta_t=1):
    """Query the MTMount elevation telemetry for a given `t_stamp`"""
    t_stamp = Time(t_stamp)
    start_time = t_stamp - timedelta(seconds=delta_t)
    end_time = t_stamp

    _df = getEfdData(
        client=client,
        topic="lsst.sal.MTMount.elevation",
        columns="actualPosition",
        begin=start_time,
        end=end_time,
    )

    return _df

## Data Analysis

Get data to search for `Fault` messages.
The `salIndex` can be either 1 (Camera Hexapod) or 2 (M2 Hexapod).
- LSSTCam data from April 20, 2025.
- ComCam data between October 24 2024 and December 10 2024 (data missing for November 1 and December 1 2024, even if there are M2 Hexapod faults).

In [ ]:
#######
# ComCam
# cc_start1 = Time("2024-10-24T00:00:00", scale="utc", format="isot")
# cc_end1 = Time("2025-11-01T00:00:00", scale="utc", format="isot")
#######
# LSSTCam
cc_start1 = Time("2025-07-01T8:00:00", scale="utc", format="isot")
cc_end1 = Time("2025-07-03T10:00:00", scale="utc", format="isot")

## Levels for different errors

|  Level  |  Numeric value  |  What it means / When to use it  |
|---    |:-:    |:-:    |
| logging.NOTSET | 0 | When set on a logger, indicates that ancestor loggers are to be consulted to determine the effective level. If that still resolves to NOTSET, then all events are logged. When set on a handler, all events are handled. |
| logging.DEBUG | 10 | Detailed information, typically only of interest to a developer trying to diagnose a problem. |
| logging.INFO | 20 | Confirmation that things are working as expected. |
|logging.WARNING | 30 | An indication that something unexpected happened, or that a problem might occur in the near future (e.g. ‘disk space low’). The software is still working as expected. |
|logging.ERROR | 40 | Due to a more serious problem, the software has not been able to perform some function. |
| logging.CRITICAL | 50 | A serious error, indicating that the program itself may be unable to continue running. |

In [ ]:
# get the Fault messages
query = f"""
SELECT
level, salIndex, message, functionName
FROM "lsst.sal.MTHexapod.logevent_logMessage"
WHERE time >= '{cc_start1.isot}Z'
AND time <= '{cc_end1.isot}Z'
AND salIndex = 2
AND level > 40


"""

M2Hexfault_df = await efd_client.influx_client.query(query)

In [ ]:
M2Hexfault_df

In [ ]:
M2Hexfault_df.values

In [ ]:
len(M2Hexfault_df)

errorCode =1 ==> 'Low-level controller went to FAULT state'\
errorCode =-2 ==> 'Compensation failed.'\
errorCode =2 is related to loss of connection, which seems to impact elevation angle data logging
- ==>'Lost connection to the low-level controller'
- ==> 'Connection refused by host=m2-hexapod-pxi.cp.lsst.org, port=5550'

In [ ]:
# Create timestamps related to messages only related to actual system faults (keep errorCode =1 and errorCode =-2)
M2Hexfault_t = np.array([])
cut = (M2Hexfault_df["message"].str.contains("errorCode=1")) | (
    M2Hexfault_df["message"].str.contains("errorCode=-2")
)
for i in M2Hexfault_df[cut].index:
    M2Hexfault_t = np.append(M2Hexfault_t, [i])

In [ ]:
M2Hexfault_t

In [ ]:
len(M2Hexfault_t)

In [ ]:
# Get elevation angle corresponding to the Fault timestamps
hexeleva = []
for i in range(len(M2Hexfault_t)):
    hexelev = query_MTMount_elevation_telemetry(efd_client, M2Hexfault_t[i], delta_t=1)
    if len(hexelev) != 0:
        hexeleva.append(hexelev["actualPosition"].iloc[-1])
    # print(M2Hexfault_t[i])
# hexeleva

In [ ]:
len(hexeleva)

Make sure the timestamps (M2Hexfault_t) and the elevation (hexeleva) arrays are of same dimensions.\
Sometimes there's missing data in elevation (ComCam)

In [ ]:
# plot elevation vs fault times
fig, ax = plt.subplots(1, 1, dpi=125, figsize=(8, 4))
ax.plot(M2Hexfault_t, hexeleva, marker="+", linestyle="")
ax.set(
    ylabel="Elevation angle (degrees)",
    xlabel="Date",
    title="Elevation angle just before M2Hex faults; level = 50; LSSTCam until July 2\n Keeping only errorCode = 1 or -2",
)
plt.xticks(rotation=90)
fig.tight_layout()

In [ ]:
# histogram
fig, ax = plt.subplots(1, 1, dpi=125, figsize=(8, 4))
binwidth = 8
ax.hist(hexeleva, bins=range(0, 100 + 10, 5))
ax.set(
    ylabel="Counts",
    xlabel="Elevation angle (degrees)",
    title="Counts TMA is at a given elevation angle just before M2Hex faults \n level = 50; LSSTCam until July 2",
)
fig.tight_layout()

## Comment

It looks like the M2 Hexapod faults more often at higher elevation, however, the TMA is more often at higher elevations than at lower elevation. It might be intersting to look at the ratio of faults at a given angle with respect to the number of times TMA is at that elevation angle. However, this requires other ways of accessing the data. 

## Looking at the actuator positions

If you are intersted in looking at the actuator positions when the M2 Hexapod goes to Fault:
## BUT BEWARE CAN TAKE TIME IF A LOT OF DATA => a few minutes for a couple of days

In [ ]:
query = f"""
SELECT
calibrated0, calibrated1, calibrated2, calibrated3, calibrated4, calibrated5
FROM "lsst.sal.MTHexapod.actuators"
WHERE time >= '{cc_start1.isot}Z'
AND time <= '{cc_end1.isot}Z'
AND salIndex = 2


"""

M2Hexpos_df = await efd_client.influx_client.query(query)

In [ ]:
M2Hexpos_df

In [ ]:
len(M2Hexpos_df)

In [ ]:
# Strut position vs time
fig, ax = plt.subplots(1, 1, dpi=125, figsize=(8, 4))
ax.plot(M2Hexpos_df, linestyle="-")
ax.set(
    ylabel="Strut position error (micrometer)",
    xlabel="Date",
    title="Strut error (6 struts) of M2 Hexapod (June 25)",
)
# ax.set_ylim(-1500, 2000)
plt.xticks(rotation=90)
ax.legend(
    [
        "strut1(i=0)",
        "strut2(i=1)",
        "strut3(i=2)",
        "strut4(i=3)",
        "strut5(i=4)",
        "strut6(i=5)",
    ],
    fontsize="xx-small",
)
# adding lines corresponding to the timestamps where M2 Hexapod goes to fault
for i in range(len(M2Hexfault_t)):
    plt.axvline(M2Hexfault_t[i], color="r", linestyle=":")
# plt.axhline(4150, color="c")
fig.tight_layout()